# DLL Lab — 26 February 2026
## R-CNN Family — Object Detection Comparison
**Done By:** Student | **Course:** AI-612


In [1]:
import torch, torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
    maskrcnn_resnet50_fpn,   MaskRCNN_ResNet50_FPN_Weights
)
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


## R-CNN Family Summary

In [2]:
models_info = [
    ('R-CNN (2014)',        0.05,  66.0,  60,   2000, 'AlexNet + SVM per region'),
    ('Fast R-CNN (2015)',   2.3,   70.0, 108,   2000, 'Shared conv features + RoI Pool'),
    ('Faster R-CNN (2015)',17.0,   73.2,  42,    300, 'RPN replaces selective search'),
    ('Mask R-CNN (2017)',   11.0,  37.1,  44,    300, 'Adds pixel-level mask branch'),
]
print(f"{'Model':22s} {'FPS':>6} {'mAP':>6} {'Params':>8} {'Proposals':>10}")
print('-'*60)
for name,fps,mmap,params,props,note in models_info:
    print(f'{name:22s} {fps:>6.1f} {mmap:>6.1f} {params:>8}M {props:>10}')
    print(f'  -> {note}')


Model                    FPS    mAP   Params  Proposals
------------------------------------------------------------
R-CNN (2014)            0.1   66.0       60M       2000
  -> AlexNet + SVM per region
Fast R-CNN (2015)       2.3   70.0      108M       2000
  -> Shared conv features + RoI Pool
Faster R-CNN (2015)    17.0   73.2       42M        300
  -> RPN replaces selective search
Mask R-CNN (2017)      11.0   37.1       44M        300
  -> Adds pixel-level mask branch


## Load Faster R-CNN and Mask R-CNN

In [3]:
frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT).to(device)
mrcnn = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT).to(device)
frcnn.eval(); mrcnn.eval()
print(f'Faster R-CNN params: {sum(p.numel() for p in frcnn.parameters()):,}')
print(f'Mask R-CNN   params: {sum(p.numel() for p in mrcnn.parameters()):,}')


Faster R-CNN params: 41,755,286
Mask R-CNN   params: 44,401,242


In [4]:
names  = ['R-CNN','Fast R-CNN','Faster R-CNN','Mask R-CNN']
fps_v  = [0.05, 2.3, 17.0, 11.0]
map_v  = [66.0, 70.0, 73.2, 37.1]
colors = ['#e74c3c','#f39c12','#27ae60','#3498db']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bars = axes[0].bar(names, fps_v, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Inference Speed (FPS)', fontsize=12)
axes[0].set_ylabel('FPS'); axes[0].grid(True, axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=15)
for bar, v in zip(bars, fps_v):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v}', ha='center', fontsize=10)

axes[1].scatter(fps_v, map_v, c=colors, s=200, zorder=5, edgecolors='black')
for nm, fx, fy in zip(names, fps_v, map_v):
    axes[1].annotate(nm, (fx, fy), xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[1].set_xlabel('FPS'); axes[1].set_ylabel('mAP (%)')
axes[1].set_title('Speed vs Accuracy', fontsize=12)
axes[1].grid(True, alpha=0.3)
plt.suptitle('R-CNN Family Comparison', fontsize=13)
plt.tight_layout(); plt.show()


## Summary
| Model | FPS | mAP | Key Innovation |
|---|---|---|---|
| R-CNN | 0.05 | 66% | Region proposals + CNN |
| Fast R-CNN | 2.3 | 70% | Shared feature map |
| Faster R-CNN | 17 | 73.2% | Region Proposal Network |
| Mask R-CNN | 11 | 37.1% | + Segmentation mask |
